In [ ]:
import kaggle_benchmarks as kbench

import json
import re
import math
from datetime import datetime

def extract_json(text):
    if not text: return None
    fence = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if fence:
        blob = fence.group(1)
    else:
        start, end = text.find("{"), text.rfind("}")
        if start == -1 or end == -1 or end <= start: return None
        blob = text[start:end + 1]
    try: return json.loads(blob)
    except: return None

def numeric_pass(answer_text, ground_truth, rel_tol=0.015):
    def parse_physics_number(text):
        s = str(text).replace(",", "").strip().lower()
        s = re.sub(r"\\times\s*10\s*(\^|e)\s*{{?(-?\d+)}}?", r"e\2", s)
        s = re.sub(r"\*\s*10\s*(\^|e)\s*{{?(-?\d+)}}?", r"e\2", s)
        match = re.search(r"[-+]?\d*\.?\d+(?:[eE^][-+]?\d+)?", s)
        if match:
            try: return float(match.group(0).replace("^", "e"))
            except: return None
        return None
    
    pred = parse_physics_number(answer_text)
    try:
        target = float(ground_truth)
        if pred is None: return False
        if target == 0: return abs(pred) < 1e-9
        return math.isclose(pred, target, rel_tol=rel_tol)
    except: return False

def build_trace(*, task_id, llm, prompt, response, parsed, final_answer, passed, failure_mode):
    return {
        "timestamp_utc": datetime.utcnow().isoformat() + "Z",
        "task_id": task_id,
        "model": str(llm),
        "pass": bool(passed),
        "failure_mode": failure_mode,
        "final_answer": final_answer,
        "raw_output": response,
        "parsed_output": parsed,
        "prompt": prompt
    }

# ----------------------------
# Task 11: Finite Chain Detachment Expectation
# ----------------------------
TASK_ID = "fp_11"
GROUND_TRUTH = 4.84

@kbench.task(name="FP-11 Finite Chain Detachment Expectation", description="Physics")
def task_11(llm) -> tuple[int, int]:
    prompt = """You are solving a frontier physics problem. Return valid JSON only.\n\nA polymer is confined to a narrow planar trench and follows a prescribed right-angle “staircase” of N=20 unit segments (d=1.00). Its rightmost endpoint is fixed at P_0 = (8.00, 0.00). Moving inward along the chain from this end, the segment directions alternate between a unit step in -x and a unit step in +y (starting with -x for the first segment). Segments can detach only in order from the right end, so at any instant the set of detached segments is a single contiguous block of length n in {0,1,...,20}. Detaching a segment costs energy epsilon = 1.70. Each detached segment has g=3 internal microstates, whereas each attached segment has one. The two free ends are joined by a frictionless ring connected to an ideal force source that maintains a constant tension of magnitude f=0.70 along the direction of the taut free polymer. The ring’s location is fixed at a clamp point A = (15.00, 0.00). For a given n, the free polymer between A and the peel front is taut and takes the shortest possible path in the plane from A to the peel front that does not enter a smooth circular exclusion region (a post) of radius R = 1.20 centered at (10.00, 3.50). What is the expected number of detached segments <n>, in thermal equilibrium for this finite chain?\n\nReturn JSON: {\"final_answer\": \"<value>\"}"""
    response = llm.prompt(prompt)
    parsed = extract_json(response)
    final_ans = parsed.get("final_answer", "") if parsed else ""
    passed = numeric_pass(final_ans, GROUND_TRUTH)
    return (1 if passed else 0, 1)


In [ ]:
task_11.run(kbench.llm)